# $a_1$--$a_2$ prior and posterior contours for every suite

This notebook is the only producer of the $(a_1, a_2)$ coefficient-contour comparison; the per-suite `postfit_physical_parameters.ipynb` notebooks no longer draw it. Edit the configuration cell to choose the suites, priors, and posteriors, then run all cells: one figure is drawn and saved per suite. Dashed contours are priors; solid contours are posteriors. Legend entries follow the standard format, "MINERvA (2026) prior, $k_{\max}=6$" for priors and "Posterior, $k_{\max}=6$" for posteriors, which becomes "Posterior from uniform prior, $k_{\max}=6$" when more than one prior is drawn.

**Spline-factorization correction.** Every posterior quantity below is computed with the importance weights $w_k=\exp(+\Delta\chi^2_{\rm data}(\eta^{(k)})/2)$ that correct PROfit's multiplicative combination of the one-dimensional PCA splines to the exact z-expansion response (`python/scripts/spline_reweighting.py`, grids from `12_spline_factorization_validation.ipynb`, demonstration in `13_spline_reweighting_comparison.ipynb`). The weights are applied inside `postfit_physical_parameters.load_fit` and propagate to the summary tables, covariances, credible bands and corner plots; the effective sample size after reweighting is printed when each chain is loaded. The correction is negligible for the LQCD-constrained and Gaussian MINERvA priors and matters only for the uniform-prior fits.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython.display import display

repo = Path.cwd().resolve()
while repo.name != "axial_mass" and repo != repo.parent:
    repo = repo.parent
if repo.name != "axial_mass":
    raise RuntimeError("Run this notebook from within the axial_mass repository")
helper_dir = repo / "ma_zexp" / "python" / "scripts"
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

from postfit_physical_parameters import (
    FIGURE_ROOT, REFERENCE_PRIORS, SPECS, SUITE_DATA_DIRS, load_fit,
    plot_distribution_overlay,
)
from spline_reweighting import describe_ess

## Configuration

`SUITES` lists the fit suites to loop over; each one gets its own figure under `figs/<suite>/comparison_overlays/`. `PRIORS` and `POSTFITS` are independent, so either list may be empty. Standalone prior keys are `deuterium`, `deuterium_k6`, `minerva_k6`, `lqcd_k6`, and `minerva_lqcd_k6`; any z-expansion fit key listed by the next cell can also supply its fitted prior. Posterior keys must exist in every selected suite. Use `deuterium_k6`, rather than native-basis `deuterium`, when comparing with the other $k_{\max}=6$ distributions.


In [ ]:
# Every suite gets its own figure. Remove entries to run a subset.
SUITES = [
    "nuwro_fit_results",
    "asimov_fit_results",
    "opendata_fit_results",
]

# Dashed contours.
PRIORS = [
    "deuterium_k6",
    "minerva_k6",
    "lqcd_k6",
    "minerva_lqcd_k6",
]

# Solid contours (posteriors). These are loaded from the MCMC output of each suite.
POSTFITS = [
    "minerva_k6_uniform",
]

BURN_IN = 0
THIN = 1
N_PRIOR_SAMPLES = 100_000
BINS = 55
FIGSIZE = (7.0, 6.2)
SAVE_FIGURE = True
OUTPUT_STEM = "a1_a2_prior_postfit_contours"
SAVE_DPI = 600

In [ ]:
standalone_prior_keys = (
    "deuterium", "deuterium_k6", *REFERENCE_PRIORS.keys(),
)
fit_specs = {spec.key: spec for spec in SPECS if spec.prior is not None}
print("Known suites:", ", ".join(SUITE_DATA_DIRS))
print("Standalone priors:", ", ".join(standalone_prior_keys))
print("Z-expansion fit keys:", ", ".join(fit_specs))

## Validate the selection

The selection is checked once, before any suite is loaded. A selected fit key is loaded once per suite even if both its prior and post-fit distribution are requested. Standalone priors are sampled directly and do not require fit output.


In [ ]:
unknown_postfits = sorted(set(POSTFITS) - set(fit_specs))
unknown_priors = sorted(
    set(PRIORS) - set(fit_specs) - set(standalone_prior_keys)
)
if unknown_postfits:
    raise KeyError(f"Unknown posterior key(s): {unknown_postfits}")
if unknown_priors:
    raise KeyError(f"Unknown prior key(s): {unknown_priors}")
if not SUITES:
    raise ValueError("Select at least one suite")
if not PRIORS and not POSTFITS:
    raise ValueError("Select at least one prior or posterior distribution")

# Fit priors need loading only when they do not also exist as standalone references.
fit_prior_keys = set(PRIORS) - set(standalone_prior_keys)
keys_to_load = sorted(set(POSTFITS) | fit_prior_keys)
selections = (
    [(key, "prior") for key in PRIORS]
    + [(key, "posterior") for key in POSTFITS]
)
print("Fits to load per suite:", ", ".join(keys_to_load) or "none")

## Load, draw, and save one figure per suite

A suite is skipped, with a message, when any requested fit has no unique PROfile ROOT file there.


In [ ]:
for suite in SUITES:
    print(f"\n=== {suite} ===")
    results = {
        key: load_fit(
            fit_specs[key], suite, burn_in=BURN_IN, thin=THIN,
            n_prior=N_PRIOR_SAMPLES,
        )
        for key in keys_to_load
    }
    missing = sorted(key for key, result in results.items() if result is None)
    if missing:
        print(f"Skipping {suite}: no unique PROfile ROOT file found for {missing}")
        continue
    for key, result in results.items():
        print(f"Loaded {key}: {len(result['samples']):,} posterior samples; "
              + describe_ess(result['ess'], len(result['samples'])))

    figure = plot_distribution_overlay(
        results, selections, bins=BINS,
        n_reference_samples=N_PRIOR_SAMPLES, figsize=FIGSIZE,
    )
    if SAVE_FIGURE:
        output_dir = FIGURE_ROOT / suite / "comparison_overlays"
        output_dir.mkdir(parents=True, exist_ok=True)
        for extension in ("pdf",):
            output_path = output_dir / f"{OUTPUT_STEM}.{extension}"
            figure.savefig(
                output_path, dpi=SAVE_DPI, bbox_inches="tight",
                pad_inches=.03, facecolor="white",
            )
            print("Saved:", output_path)
    display(figure)
    plt.close(figure)